# 讓 Agent 先找到依據再回答
metadata

## 在 Colab 準備環境

如果你是在 Colab 開啟，先複製專案並切換到專案資料夾；如果你已經在專案資料夾，可以直接跳過這格。

In [ ]:
!git clone https://github.com/R300-AI/Agentic-SDK.git
%cd Agentic-SDK

## 載入流程元件

這一章用 `KeywordRetrieve` 做最小示範，因為它不需要外部索引或模型服務。重點不是 keyword 本身，而是 retrieve 階段把「可引用的依據」交給後續 action。

In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceive

## 準備一小組可引用的依據

先把參考資料寫成幾筆明確條目。這些條目代表 Agent 可以拿來回答的資料邊界；如果問題不在這些資料裡，就應該進入查不到資料的降級路徑。

In [ ]:
reference_items = [
    {'keywords': ['保存', 'bundle', '參考文件'], 'content': '使用參考文件的 Agent 需要保存 bundle，重新打開時才找得到原本的資料。'},
    {'keywords': ['runner', '唯讀'], 'content': '公開分享的 Runner 可以使用 Agent，但不能修改設定或保存。'},
    {'keywords': ['builder', '建立'], 'content': 'Builder 用來建立或調整 Agent 設定，再交給 Runner 試跑。'},
]

reference_items

## 定義查詢契約

`retrieve` 這個節點的責任是把使用者問題對應到可引用內容。這裡另外設定 fallback，讓「沒有找到依據」也成為明確、可測的流程結果。

In [ ]:
retrieve = KeywordRetrieve(
    items=reference_items,
    fallback='目前沒有找到相關參考資料。',
)

retrieve

## 組成先查資料再回答的流程

`Workflow` 不需要知道資料來自 keyword、語意搜尋或向量索引。它只需要一個 retrieve 節點提供可用 context，再由 action 使用這份 context 產生回答。

In [ ]:
workflow = Workflow(
    workflow_name='參考資料問答 Agent',
    perceive=PassThroughPerceive(),
    retrieve=retrieve,
    action=DirectAnswerAction(),
)

## 第一種情況：問題命中參考資料

這個問題會命中 bundle 相關條目。最後回答應該來自查回來的內容，而不是憑空補充。

In [ ]:
grounded_result = workflow.run('為什麼使用參考文件的 Agent 要保存 bundle？')
print(grounded_result.final_message)

## 檢查回答背後的 evidence

如果最後回答不對，第一件事不是改提示詞，而是先看 retrieve 是否提供了正確依據。`latest_retrieved_content` 是這個最小流程裡 action 會讀取的 evidence 欄位。

In [ ]:
print('查回來的依據:', grounded_result.entities.get('latest_retrieved_content'))
print('完整 entities:', grounded_result.entities)

## 第二種情況：沒有找到依據

同一個 workflow 也要能處理查不到資料的問題。這時候重點不是硬回答，而是讓 fallback 成為可觀察、可測試的產品行為。正式接上 `SemanticRetrieve`、文件 bundle 或向量索引時，這個契約仍然不變：retrieve 提供 evidence，action 依據 evidence 回答；沒有 evidence，就走明確的降級路徑。

In [ ]:
missing_result = workflow.run('這個 Agent 支援哪些 GPU driver 版本？')
print(missing_result.final_message)
print('查回來的依據:', missing_result.entities.get('latest_retrieved_content'))